# Experiment: Independent Cohort Webcam vs Professional Gaze Comparison

**Question.** When the Webcam/Prolific and professional/Mayo samples contain 500 different participants each, are their workflow-and-cohort distributions sufficiently similar for a named group-level gaze endpoint?

> **100% synthetic, research-only, and nonclinical.** This notebook contains no observed Mayo or Prolific data and cannot isolate a pure device effect or establish individual interchangeability.


## Success criteria defined before the mock run

1. Common stimulus, task, transform, and QC gates pass.
2. Technical endpoint 90% CIs are compared with explicit illustrative margins.
3. Cross-domain group-map similarity is compared with within-cohort split-half repeatability.
4. Technical and attention-pattern source-domain AUCs are reported separately.

No single overall score is created. The mock margins demonstrate code behavior; they are not Mayo acceptance thresholds.


In [ ]:
from __future__ import annotations

from pathlib import Path
from IPython.display import Image, display

from gaze_compare.cohort_analysis import run_independent_cohort_analysis
from gaze_compare.cohort_report import write_independent_outputs
from gaze_compare.cohort_simulate import load_cohort_config, simulate_independent_cohorts

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'config' / 'mock_independent_study.json').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
CONFIG_PATH = PROJECT_ROOT / 'config' / 'mock_independent_study.json'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebook_demo'
config = load_cohort_config(CONFIG_PATH)
{key: config[key] for key in ['project_id', 'seed', 'participants_per_cohort', 'stimulus_ids']}


## Generate two independent mock cohorts

Participant IDs are disjoint by construction. The simulation intentionally contains modest technical differences and much smaller attention-pattern differences so the two domain questions can disagree.


In [ ]:
inputs = simulate_independent_cohorts(config)
participants = inputs['participants']
participants.groupby('device').agg(
    participants=('participant_id', 'nunique'),
    mean_accuracy_deg=('accuracy_deg', 'mean'),
    mean_data_loss=('data_loss', 'mean'),
)


In [ ]:
webcam_ids = set(participants.loc[participants.device.eq('webcam'), 'participant_id'])
professional_ids = set(participants.loc[participants.device.eq('professional'), 'participant_id'])
assert webcam_ids.isdisjoint(professional_ids)
assert len(webcam_ids) == len(professional_ids) == 500
'Confirmed: 500 + 500 different participants; no pairing.'


## Run the tested independent-cohort pipeline

The map bootstrap resamples participants, not fixation rows. The source classifier uses repeated stratified cross-validation and never receives site or recruitment labels.


In [ ]:
result = run_independent_cohort_analysis(
    inputs['participants'], inputs['fixations'], inputs['aois'], config=config
)
manifest_path = write_independent_outputs(result, OUTPUT_DIR, config=config)
result.tables['decision_summary']


## Primary numerical results

Read the quality table as practical similarity intervals, not ordinary null-hypothesis tests. Read map SIM relative to the within-cohort sampling benchmark.


In [ ]:
result.tables['quality_comparison'][[
    'label', 'webcam_mean', 'professional_mean',
    'mean_difference_webcam_minus_professional', 'ci90_lower', 'ci90_upper',
    'equivalence_margin', 'decision'
]]


In [ ]:
result.tables['map_reliability'][[
    'stimulus_id', 'cross_domain_similarity', 'within_webcam_similarity',
    'within_professional_similarity', 'cross_minus_lower_within', 'decision'
]]


In [ ]:
result.tables['domain_classifier'][[
    'feature_set', 'auc', 'ci95_lower', 'ci95_upper', 'interpretation'
]]


## Three figures that answer the main question

The complete output directory contains seven figures. These three show practical quality similarity, map repeatability, and feature-space domain separability.


In [ ]:
for filename in [
    '02_quality_equivalence.png',
    '05_map_reproducibility.png',
    '07_domain_classifier.png',
]:
    display(Image(filename=str(OUTPUT_DIR / 'figures' / filename), width=900))


## Interpretation and next step

A defensible conclusion is endpoint-specific: group-level maps may be close to sampling repeatability even when technical features remain distinguishable. This does **not** establish individual-level agreement or a pure device effect.

Before real-data analysis, freeze common stimulus/task/transform/QC versions and replace every illustrative margin with a clinician- and methods-justified preregistered value. If individual interchangeability becomes the goal, collect both streams on the same people.
